# Enterprise RAG — Hands-On, Part 11 of 11: Observability, and what to take away

*Split from `02-hands-on.ipynb` for focused reading — same content, one phase at a time. The
"Setup" cell below re-derives whatever state earlier parts would have produced, so this notebook
runs standalone; you do not need to run the other parts first.*

**Prerequisites:** `OPENAI_API_KEY` in the repo-root `.env`, and `python scripts/ingest.py` already
run (the setup cell below will build the index for you if it is missing).

**Series:** [1. The corpus and its permissions](part01-corpus-and-permissions.ipynb) · [2. The policy engine](part02-policy-engine.ipynb) · [3. Compiling the policy into a database filter](part03-compiling-policy-to-filter.ipynb) · [4. Chunking and ingestion](part04-chunking-and-ingestion.ipynb) · [5. Why hybrid search, demonstrated](part05-hybrid-search.ipynb) · [6. Query transformation](part06-query-transformation.ipynb) · [7. Reranking](part07-reranking.ipynb) · [8. The full graph](part08-full-graph.ipynb) · [9. Attacking it](part09-attacking-it.ipynb) · [10. Evaluation](part10-evaluation.ipynb) · [11. Observability, and what to take away](part11-observability-and-takeaways.ipynb)

---


In [1]:
import sys, json, textwrap
from pathlib import Path

# The package lives in src/ - add it to the path so this notebook runs from anywhere.
ROOT = Path.cwd()
while not (ROOT / "src" / "enterprise_rag").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from enterprise_rag.config import SETTINGS

print("project root :", ROOT)
print("corpus       :", SETTINGS.corpus_dir.relative_to(ROOT))
print("api key      :", "found" if SETTINGS.has_api_key else "MISSING - check .env")
print("embed model  :", SETTINGS.embedding_model)
print("chat model   :", SETTINGS.fast_model)

project root : d:\INTERVIEW PREPARATION\DevRev_Preparation\enterprise_rag_platform
corpus       : data\corpus
api key      : found
embed model  : text-embedding-3-small
chat model   : gpt-4o-mini


### Setup — recap of state from earlier parts


In [2]:
import textwrap
from enterprise_rag.identity import get_principal
from enterprise_rag.graph.build import RAGPlatform

TODAY = "2026-08-22"

# Build the index if it is not already there (same check as Part 4).
from enterprise_rag.ingest.store import collection_stats
try:
    stats = collection_stats("meridian")
    assert stats["chunks"] > 0
except Exception:
    from enterprise_rag.ingest.pipeline import ingest
    print("building index...")
    print(ingest().render())

platform = RAGPlatform()

def ask(user_id, question, strategy="enterprise", as_of=TODAY, show=True):
    p = get_principal(user_id)
    res = platform.ask(question, p, strategy=strategy, as_of=as_of, write_trace=False)
    a, t = res["answer"], res["trace"]
    if show:
        print(f"[{p.role}]  strategy={strategy}"
              f"{'  REFUSED' if a.refused else ''}")
        print(textwrap.fill(a.text, 92))
        print(f"\ncited: {[c.doc_id for c in a.citations] or 'none'}")
        if t.denied:
            rules = sorted({d['rule'] for d in t.denied})
            print(f"denied: {sorted({d['doc_id'] for d in t.denied})} by {rules}")
        if t.redacted_count:
            print(f"redacted: {t.redacted_count} chunk(s)")
        print(f"groundedness={t.groundedness}  {t.total_ms:.0f}ms  ${t.cost_usd:.5f}")
    return res

---
# Part 11 - Observability

Every run produces a complete, replayable record: who asked, what the policy decided, which queries
were generated, what was retrieved and why, what was denied and by which rule, what the model saw,
what it produced, how long each stage took, and what it cost.

Three audiences, one artefact: the **engineer** debugging a bad answer, the **auditor** asking "did
this user ever see that document?", and the **finance team** asking "which tenant is burning the
budget?".

In [3]:
res = ask("u_sofia_am",
          "What service credit does Vertex get if availability drops to 99.2%, "
          "and what caused last March's incident?",
          strategy="enterprise", show=False)
tr = res["trace"]

print(textwrap.fill(res["answer"].text, 92))
print("\n" + "=" * 78)
print(tr.timeline())
print("\naccess filter :", tr.prefilter_explained)
print("prompt version:", tr.prompt_version)
print("groundedness  :", tr.groundedness)
print("\nretrieval fan-out:")
for q in tr.generated_queries:
    print("   variant:", q)
for s in tr.subquestions:
    print("   sub-q  :", s)
print("\ncontext shown to the model:")
for c in tr.candidates:
    print(f"   {c['chunk_id']:<18}{c['source']:<11}{c['sensitivity']:<13}"
          f"rerank={c['rerank']}  via={'+'.join(c['retrieved_by'])}")
print("\naudit events (confidential reads):")
for e in tr.audit_events:
    print("  ", e)

If availability drops to 99.2%, Vertex is entitled to a **25%** service credit against the
following month's fees, as it falls below 99.5% but at or above 99.0% [CT-VTX-001]. The
cause of last March's incident is not mentioned in the available material.

run 9d954567ef95  |    13793 ms total  |  $0.00113  |  8 llm calls
  authorize            0 ms  #
  plan              2295 ms  ######
  retrieve          4274 ms  ############
  enforce              2 ms  #
  rerank            4381 ms  ############
  grade              793 ms  ##
  generate          1282 ms  ###
  verify             754 ms  ##

access filter : tenant=meridian AND sensitivity<=confidential AND region in (GLOBAL,EU) AND groups overlap (sales, account-management)
prompt version: 2026-08-22.v3
groundedness  : 1.0

retrieval fan-out:
   variant: What service credit does Vertex get if availability drops to 99.2%, and what caused last March's incident?
   variant: Vertex service credit policy for availability below 99.2%
   v

In [4]:
# Token and cost attribution, broken down by what the tokens were spent ON.
# This is what makes "which tenant is burning the budget" answerable.
print(f"total cost   : ${tr.cost_usd:.5f}")
print(f"llm calls    : {tr.llm_calls}")
print(f"tokens in/out: {tr.tokens_in} / {tr.tokens_out}   embedding: {tr.embedding_tokens}")
print("\nper-stage token spend:")
usage = res["state"]["usage"]
for purpose, toks in sorted(usage.by_purpose.items(), key=lambda kv: -kv[1]):
    print(f"   {purpose:<16}{toks:>7} tokens")

total cost   : $0.00113
llm calls    : 8
tokens in/out: 4139 / 847   embedding: 286

per-stage token spend:
   rerank             1977 tokens
   synthesis           981 tokens
   grade               675 tokens
   groundedness        635 tokens
   hyde                315 tokens
   embed_query         286 tokens
   multi_query         242 tokens
   decompose           161 tokens


---
# What to take away

1. **Access control decides the architecture.** Pre-filter inside the vector search; partition by
   tenant on top. Never post-filter, never let the model enforce anything.
2. **Two layers.** The compiled filter makes retrieval cheap; the post-retrieval policy re-check makes
   it correct - and catches stale indexes, embargoes, compartments, and live revocation.
3. **Hybrid + RRF is the baseline, not the advanced option.** Enterprise text is full of identifiers
   that embeddings handle badly.
4. **Rerank after enforcement.** The user's top-k should be the best of *their* pool.
5. **Refusing well is a feature.** Partial answers where the role allows; clean escalation where it
   does not; never hint that withheld material exists.
6. **The leak test is a gate, not a metric.** Zero, or the release does not ship.

Next: `../INTERVIEW_SCRIPT.md` - how to present all of this on a whiteboard in 60 minutes.

---

**◀ Previous:** [10. Evaluation](part10-evaluation.ipynb)

That's the whole series. See `README.md` in this folder for the index, or `../../docs/06-architecture-end-to-end.md` for the full architecture reference, or `../../INTERVIEW_SCRIPT.md` for the 60-minute whiteboard script.
